In [ ]:
%pip install -q kagglehub
import kagglehub, os, shutil, random, glob

path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
roots = glob.glob(os.path.join(path, '**', 'chest_xray'), recursive=True)
src_base = roots[0] if roots else path

dst_base = "chest_xray_small"
random.seed(42)
for split in ["train", "test"]:
    n = 50 if split == "train" else 25
    for cls in ["NORMAL", "PNEUMONIA"]:
        s = os.path.join(src_base, split, cls)
        d = os.path.join(dst_base, split, cls)
        os.makedirs(d, exist_ok=True)
        if os.path.isdir(s):
            files = [f for f in os.listdir(s) if f.lower().endswith(('.jpg','.jpeg','.png'))]
            sel = random.sample(files, min(n, len(files)))
            for f in sel:
                shutil.copy2(os.path.join(s, f), os.path.join(d, f))
print("Trimmed chest x-ray dataset ready")

In [ ]:
import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
train_dir="chest_xray_small/train"
test_dir="chest_xray_small/test"



train_datagen = ImageDataGenerator(
    rescale=1./255, 
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

test_datagen = ImageDataGenerator(rescale=1./255)

In [ ]:
train_func = train_datagen.flow_from_directory(
    train_dir, 
    target_size=(224, 224), 
    batch_size=32,
    class_mode='binary' 
)


test_func = test_datagen.flow_from_directory(
    test_dir, 
    target_size=(224, 224), 
    batch_size=32,
    class_mode='binary' 
)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Flatten, Dense, Dropout
from tensorflow.keras.applications import ResNet50

In [ ]:
resnet = ResNet50(weights='imagenet', input_shape=(224, 224, 3))

x = Flatten()(resnet.output) 
x = Dense(128, activation='relu')(x)  
x = Dropout(0.5)(x)  
x = Dense(1, activation='sigmoid')(x) 

resnet_model = Model(inputs=resnet.input, outputs=x)

resnet_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4), loss='binary_crossentropy', metrics=['AUC'])

In [ ]:
history = resnet_model.fit(
    train_func, 
    epochs=5, 
    validation_data=test_func  
)

In [ ]:
train_func = train_datagen.flow_from_directory(
    train_dir, 
    target_size=(224, 224), 
    batch_size=16,
    class_mode='binary' 
)


test_func = test_datagen.flow_from_directory(
    test_dir, 
    target_size=(224, 224), 
    batch_size=16,
    class_mode='binary' 
)

In [ ]:
resnet = ResNet50(weights='imagenet', input_shape=(224, 224, 3))

x = Flatten()(resnet.output) 
x = Dense(128, activation='relu')(x)  
x = Dropout(0.5)(x)  
x = Dense(1, activation='sigmoid')(x) 

resnet_model = Model(inputs=resnet.input, outputs=x)

resnet_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4), loss='binary_crossentropy', metrics=['AUC'])

history = resnet_model.fit(
    train_func, 
    epochs=5, 
    validation_data=test_func  
)


In [ ]:
resnet_model.save("ass2.h5")

In [ ]:
resnet_model.compile(optimizer='sgd', loss='hinge', metrics=['accuracy'])

In [ ]:
history = resnet_model.fit(
    train_func, 
    epochs=5, 
    validation_data=test_func  
)

In [ ]:
resnet_model.save("ass2-1.h5")

In [ ]:
resnet_model.compile(optimizer='sgd', loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
history = resnet_model.fit(
    train_func, 
    epochs=5, 
    validation_data=test_func  
)